# Week 2 Exercise — Multi-Model Technical Q&A Assistant

Builds a full prototype of the technical Q&A tool from week 1, now with:
- A Gradio UI
- Streaming responses
- A model dropdown to switch between GPT, Claude, and a local Ollama model
- A system prompt that establishes 'senior software engineer' expertise

Useful as a personal study companion — paste in any code snippet or technical question.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

for name, key in [('OpenAI', openai_api_key), ('Anthropic', anthropic_api_key)]:
    print(f"{name}: {'set' if key else 'not set'}")

In [ ]:
openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")
ollama = OpenAI(api_key='ollama', base_url="http://localhost:11434/v1")

In [ ]:
MODEL_GPT = 'gpt-4o-mini'
MODEL_CLAUDE = 'claude-haiku-4-5'
MODEL_LLAMA = 'llama3.2'

system_message = (
    "You are a senior software engineer who explains technical concepts clearly and concisely. "
    "When given a code snippet or question, explain what it does, why it works that way, "
    "and any subtle behavior the reader should be aware of. Respond in markdown."
)

## Per-provider streaming functions

In [ ]:
def stream_gpt(question):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question}
    ]
    stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
def stream_claude(question):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question}
    ]
    stream = anthropic.chat.completions.create(model=MODEL_CLAUDE, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
def stream_llama(question):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question}
    ]
    stream = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

## Router

In [ ]:
def stream_model(question, model):
    if model == "GPT":
        result = stream_gpt(question)
    elif model == "Claude":
        result = stream_claude(question)
    elif model == "Llama":
        result = stream_llama(question)
    else:
        raise ValueError("Unknown model")
    yield from result

## Gradio UI

In [ ]:
default_question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
question_input = gr.Textbox(
    label="Your technical question:",
    info="Paste a code snippet or ask anything technical",
    lines=10,
    value=default_question
)
model_selector = gr.Dropdown(["GPT", "Claude", "Llama"], label="Select model", value="GPT")
answer_output = gr.Markdown(label="Explanation:")

view = gr.Interface(
    fn=stream_model,
    title="Technical Q&A Assistant",
    inputs=[question_input, model_selector],
    outputs=[answer_output],
    flagging_mode="never"
)
view.launch()